In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
from torchvision.models import MobileNet_V3_Large_Weights
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt

DATA_DIR   = 'dataset'
TRAIN_DIR  = os.path.join(DATA_DIR, 'BelgiumTSC_Training', 'Training')
TEST_DIR   = os.path.join(DATA_DIR, 'BelgiumTSC_Testing', 'Testing')

# MobileNetV3 is lightweight â€” larger batch and more epochs help squeeze accuracy
BATCH_SIZE  = 64
NUM_EPOCHS  = 30
NUM_CLASSES = 62
VAL_SPLIT   = 0.2
SEED        = 42
CKPT_PATH   = 'best_mobilenetv3_btsd.pth'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')
torch.manual_seed(SEED)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Transforms defined.')

In [ ]:
class NumericImageFolder(torchvision.datasets.ImageFolder):
    """ImageFolder that sorts class folders by integer value (robust to non-zero-padded names).
    For BelgiumTSC the folders are '00000'..'00061'."""
    def find_classes(self, directory):
        classes = sorted(os.listdir(directory), key=lambda x: int(x))
        class_to_idx = {cls: int(cls) for cls in classes}
        return classes, class_to_idx

full_train_dataset = NumericImageFolder(TRAIN_DIR, transform=train_transform)

# Verify mapping is correct: '00000'->0, '00001'->1, ..., '00061'->61
assert full_train_dataset.class_to_idx['00010'] == 10, "Label mapping is wrong!"
print(f"Label mapping check passed: class '00010' -> index {full_train_dataset.class_to_idx['00010']}")

n_total = len(full_train_dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val
train_dataset, val_dataset = random_split(
    full_train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
# Apply val transform to the val split
val_dataset.dataset = NumericImageFolder(TRAIN_DIR, transform=val_test_transform)

test_dataset = NumericImageFolder(TEST_DIR, transform=val_test_transform)

# num_workers=0 avoids Windows multiprocessing spawn overhead in Jupyter
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train samples: {n_train} | Val samples: {n_val} | Test samples: {len(test_dataset)}')
print(f'Classes: {NUM_CLASSES}')

imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}, Labels shape: {labels.shape}')

In [ ]:
# MobileNetV3-Large architecture:
#   features[0]     : stem conv (Conv2d + BN + Hardswish)
#   features[1-13]  : 13 InvertedResidual / SE blocks
#   features[14-16] : final conv stages
#   classifier      : Linear(960,1280) -> Hardswish -> Dropout -> Linear(1280,1000)
#
# Fine-tuning strategy:
#   Freeze features[0:12] (low-level edge/texture detectors â€” generic, waste to tune)
#   Unfreeze features[12:] (last 5 blocks including the final conv stages)
#   Unfreeze all of classifier (pretrained head weights are a useful initialisation
#   for the intermediate layers; only the last Linear gets replaced)
#
# Discriminative LR rationale:
#   features[12:]    : pretrained, but task-specific enough to benefit from gentle tuning â†’ 2e-5
#   classifier[0]    : pretrained Linear(960,1280), keep mostly frozen â†’ 5e-5
#   new_head         : randomly initialised Linear(1280,43) â†’ 5e-4

model = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

# Freeze all parameters first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the last 5 feature blocks (indices 12-16)
for block in model.features[12:]:
    for param in block.parameters():
        param.requires_grad = True

# Unfreeze entire classifier (we'll replace its last layer)
for param in model.classifier.parameters():
    param.requires_grad = True

# Replace the final classification layer (classifier[3]) with 43-class head
in_features = model.classifier[3].in_features   # 1280
model.classifier[3] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')
print(f'Output classes: {model.classifier[3].out_features}')

In [ ]:
criterion = nn.CrossEntropyLoss()

# Discriminative learning rates:
#   features[12:]  â€” pretrained conv blocks, fine-tune gently       â†’ lr 2e-5
#   classifier[0]  â€” pretrained Linear(960,1280), mild tuning       â†’ lr 5e-5
#   classifier[3]  â€” new randomly-initialised 43-class head         â†’ lr 5e-4
optimizer = torch.optim.Adam([
    {'params': [p for b in model.features[12:] for p in b.parameters()], 'lr': 2e-5},
    {'params': model.classifier[0].parameters(), 'lr': 5e-5},
    {'params': model.classifier[3].parameters(), 'lr': 5e-4},
])

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
print('Loss, optimizer (discriminative LR), and scheduler ready.')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validate ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    val_loss = running_loss / total
    val_acc  = correct / total

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    saved = val_acc > best_val_acc
    if saved:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CKPT_PATH)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}'
          + (' *** saved' if saved else ''))

print(f'\nBest val accuracy: {best_val_acc:.4f}')

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs, history['train_acc'], label='Train')
axes[1].plot(epochs, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Reload best checkpoint for evaluation
model_eval = models.mobilenet_v3_large(weights=None)
model_eval.classifier[3] = nn.Linear(model_eval.classifier[3].in_features, NUM_CLASSES)
model_eval.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model_eval = model_eval.to(device)
model_eval.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model_eval(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print()
print(classification_report(all_labels, all_preds, digits=3))

In [ ]:
import random

inv_normalize = transforms.Normalize(
    mean=[-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1/s for s in IMAGENET_STD]
)

random.seed(42)
indices = random.sample(range(len(test_dataset)), 16)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))

model_eval.eval()
with torch.no_grad():
    for ax, idx in zip(axes.flat, indices):
        img_tensor, true_label = test_dataset[idx]
        pred_label = model_eval(img_tensor.unsqueeze(0).to(device)).argmax(1).item()
        img_show = inv_normalize(img_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
        ax.imshow(img_show)
        color = 'green' if pred_label == true_label else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=9)
        ax.axis('off')

plt.suptitle('Sample Predictions (green=correct, red=wrong)', fontsize=13)
plt.tight_layout()
plt.show()